<!-- nav -->
[← 2 · Changing code safely](02_Changing_Code_Safely.ipynb) · [Home](../../README.md) · [4 · What names refer to →](04_What_Names_Refer_To.ipynb)

# 3 · Which standard?

**The problem.** The servers run Python 3.12, the developers' laptops 3.14. Code that uses a t-string runs on one and
is a syntax error on the other; a generator that emits it breaks production. Which constructs a program uses, and which
language version has them, is part of its correctness.

Every kind records the version that introduced it, and every standard checks trees against what it has.

In [1]:
from mbse.Programs.Framework import Errors
from mbse.Programs.Python import Python312, Python314, PythonStandard, Syntax as P

Py = P.LANGUAGE.Builders
modern = Python314.parse('''type Pair = tuple[int, int]


def greet(name):
    return t"hello {name}"
''')

## Step 1: checking a tree

`check(tree)` lists what a standard lacks, by path. Python 3.12 has `type` aliases, but not t-strings:

In [2]:
print(Python314.check(modern))
print(Python312.check(modern))

[]
['body[1].body[0].value: TemplateStr needs Python 3.14', 'body[1].body[0].value.values[1]: Interpolation needs Python 3.14']


## Step 2: parsing and printing refuse what a standard lacks

Each standard parses only its own language, and says where it stopped, in lines and columns:

In [3]:
try:
    Python312.parse('greeting = t"hello {name}"\n')
except Errors.ParseError as error:
    print(error)
try:
    Python312.print(modern)
except Errors.PrintError as error:
    print(error)

line 1, column 12: TemplateStr needs Python 3.14, but this is Python 3.12
body[1].body[0].value: TemplateStr needs Python 3.14, but this is Python 3.12


So a generator that prints with the standard it deploys to can't emit code that version won't run.

## Step 3: any version

`Python312` and `Python314` are the standards most code targets. `PythonStandard(major, minor)` makes any other.
Python 3.15 adds lazy imports, which a tree can hold today and 3.15 prints:

In [4]:
lazy = Py.Module().add_body(Py.Import().is_lazy(True).add_names(lambda b: b.name(lambda b: b.add_names("json")))).create()
print(Python314.check(lazy))
print(PythonStandard(3, 15).print(lazy))

['body[0]: Import.is_lazy needs Python 3.15']
lazy import json



## Step 4: features, not just kinds

Some constructs aren't kinds of their own but values of a property, like an `Import` being lazy. Kinds record those as
features, so they're checked too. The language's grammar, which every standard checks against, is data:

In [5]:
grammar = P.LANGUAGE.grammar()
kinds = {kind["kind"]: kind for kind in grammar["kinds"]}
print(len(kinds), "kinds")
print({name: kinds[name].get("since") for name in ("NamedExpr", "Match", "TypeAlias", "TemplateStr")})

86 kinds
{'NamedExpr': {'Python': 308}, 'Match': {'Python': 310}, 'TypeAlias': {'Python': 312}, 'TemplateStr': {'Python': 314}}


Versions are numbered `100 * major + minor`, so 308 is Python 3.8: a number orders a family's standards.

## Why it works this way

- **Availability is data.** Each kind and feature says where it exists, so checking a tree is a lookup, not a guess,
  and the same tree can be checked against every version a project supports.
- **One tree language for all versions.** A tree holds any construct from Python 3.0 to 3.15; standards decide what's
  allowed. A transpiler or a generator builds the tree once and asks which standard it needs.
- **Refuse, don't degrade.** Printing what a version can't run would move the error to production.

<!-- nav -->
[← 2 · Changing code safely](02_Changing_Code_Safely.ipynb) · [Home](../../README.md) · [4 · What names refer to →](04_What_Names_Refer_To.ipynb)